# CatBoost + CNN Residual Learning Pipeline

This notebook implements the workflow from `compute_instruction.md` using the existing `processed_data/train.csv`, `processed_data/validation.csv`, and `processed_data/test.csv` files. The pipeline is:

1. Load and re-sort the panel data.
2. Rebuild leak-safe lag and rolling features after concatenating the splits.
3. Train CatBoost on the train split only.
4. Compute out-of-sample residuals on validation and test.
5. Train a causal 1D CNN on validation residual sequences.
6. Evaluate CatBoost alone and the combined CatBoost + CNN model.


## 1. Imports + Config

If the notebook kernel is missing packages, run:

```python
%pip install --upgrade --no-cache-dir pandas numpy catboost torch ipython
```

If you need a CUDA-enabled Torch build, use the install command from https://pytorch.org/get-started/locally/.

In [ ]:
from pathlib import Path
import copy
import warnings

import numpy as np
import pandas as pd
import torch
from catboost import CatBoostError, CatBoostRegressor, Pool
from IPython.display import display
from torch import nn
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings("ignore")

DATA_DIR = Path("processed_data")
TARGET_COLUMN = "monthly_gross_return"
ID_COLUMN = "co_code"
DATE_COLUMN = "Month"
SOURCE_SPLIT_COLUMN = "source_split"

CAT_COLUMNS = [
    "Size_Label",
    "BM_Label",
    "OpProf_Label",
    "Inv_Label",
    "Mom_Label",
]
LAG_STEPS = (1, 3, 6)
ROLLING_WINDOWS = (3, 6)
SEQ_LEN = 6
CNN_TRAIN_FRACTION = 0.8
BATCH_SIZE = 1024
CNN_EPOCHS = 30
CNN_LR = 1e-3
WEIGHT_DECAY = 1e-4
TOP_K = 20
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#DEVICE = torch.device("cuda")
PIN_MEMORY = DEVICE.type == "cuda"

CB_PARAMS = {
    "loss_function": "RMSE",
    "eval_metric": "RMSE",
    "iterations": 1500,
    "depth": 10,
    "learning_rate": 0.05,
    "l2_leaf_reg": 3.0,
    "random_seed": SEED,
    "verbose": 100,
    "task_type": "GPU",
}

print(f"Using device: {DEVICE}")


Using device: cuda


## 2. Data Load

The files are concatenated only for feature generation and then sliced back to the original split boundaries.


In [ ]:
def read_split(path: Path, split_name: str) -> pd.DataFrame:
    frame = pd.read_csv(path, low_memory=False)
    frame[SOURCE_SPLIT_COLUMN] = split_name
    return frame


def drop_previous_engineering(df: pd.DataFrame) -> pd.DataFrame:
    explicit_drop = {
        "split_group",
        "month_index",
        "ret_lag_1",
        "ret_lag_3",
        "ret_lag_6",
        "ret_rolling_mean_3m",
        "ret_rolling_std_3m",
        "ret_rolling_mean_6m",
        "ret_rolling_std_6m",
        "ret_rolling_count_3m",
        "l2og_ret_lag_1",
        "log_ret_lag_3",
        "log_ret_lag_6",
    }
    dynamic_drop = [
        column
        for column in df.columns
        if column in explicit_drop
        or column.startswith("is_missing_")
        or column.startswith("lag_ret_")
        or column.startswith("rolling_mean_")
        or column.startswith("rolling_std_")
    ]
    return df.drop(columns=dynamic_drop, errors="ignore")


def coerce_panel_types(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df[DATE_COLUMN] = pd.to_datetime(df[DATE_COLUMN], errors="coerce")
    df[ID_COLUMN] = pd.to_numeric(df[ID_COLUMN], errors="coerce").astype("Int64")
    df[TARGET_COLUMN] = pd.to_numeric(df[TARGET_COLUMN], errors="coerce")

    for column in CAT_COLUMNS:
        if column in df.columns:
            df[column] = (
                df[column]
                .astype("string")
                .str.strip()
                .replace({"": pd.NA, "nan": pd.NA, "<NA>": pd.NA})
                .fillna("__MISSING__")
            )

    excluded = {ID_COLUMN, TARGET_COLUMN, DATE_COLUMN, SOURCE_SPLIT_COLUMN, *CAT_COLUMNS}
    for column in df.columns:
        if column not in excluded:
            df[column] = pd.to_numeric(df[column], errors="coerce")

    return df.sort_values([ID_COLUMN, DATE_COLUMN], kind="mergesort").reset_index(drop=True)


def collapse_duplicate_company_months(df: pd.DataFrame):
    duplicate_key_mask = df.duplicated([ID_COLUMN, DATE_COLUMN], keep=False)
    duplicate_key_count = int(
        df.loc[duplicate_key_mask, [ID_COLUMN, DATE_COLUMN]].drop_duplicates().shape[0]
    )
    duplicate_rows_removed = int(df.duplicated([ID_COLUMN, DATE_COLUMN], keep="last").sum())

    if duplicate_rows_removed:
        print(
            f"Collapsing {duplicate_rows_removed} duplicate rows across {duplicate_key_count} "
            f"({ID_COLUMN}, {DATE_COLUMN}) keys by keeping the last occurrence per key."
        )

    deduped = df.drop_duplicates([ID_COLUMN, DATE_COLUMN], keep="last")
    deduped = deduped.sort_values([ID_COLUMN, DATE_COLUMN], kind="mergesort").reset_index(drop=True)
    return deduped, duplicate_key_count, duplicate_rows_removed


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    grouped = df.groupby(ID_COLUMN, sort=False)
    target_history = grouped[TARGET_COLUMN]
    past_return = target_history.shift(1)

    for lag in LAG_STEPS:
        df[f"lag_ret_{lag}"] = target_history.shift(lag)

    for window in ROLLING_WINDOWS:
        rolling = past_return.groupby(df[ID_COLUMN]).rolling(window=window, min_periods=1)
        df[f"rolling_mean_{window}"] = rolling.mean().reset_index(level=0, drop=True)
        df[f"rolling_std_{window}"] = rolling.std().reset_index(level=0, drop=True)

    if "log_ret" in df.columns:
        for lag in LAG_STEPS:
            df[f"log_ret_lag_{lag}"] = grouped["log_ret"].shift(lag)

    df["month_index"] = df[DATE_COLUMN].dt.year * 12 + df[DATE_COLUMN].dt.month
    return df


def add_missing_indicators(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for column in list(df.columns):
        if column in {ID_COLUMN, DATE_COLUMN, SOURCE_SPLIT_COLUMN}:
            continue
        indicator = f"is_missing_{column}"
        if indicator not in df.columns:
            df[indicator] = df[column].isna().astype("int8")
    return df


train_raw = read_split(DATA_DIR / "train.csv", "train")
validation_raw = read_split(DATA_DIR / "validation.csv", "validation")
test_raw = read_split(DATA_DIR / "test.csv", "test")

panel = pd.concat([train_raw, validation_raw, test_raw], ignore_index=True, sort=False)
panel = drop_previous_engineering(panel)
panel = coerce_panel_types(panel)
panel, duplicate_key_count, duplicate_rows_removed = collapse_duplicate_company_months(panel)
panel = engineer_features(panel)
panel = add_missing_indicators(panel)

split_frames = {
    split_name: panel.loc[panel[SOURCE_SPLIT_COLUMN] == split_name].copy().reset_index(drop=True)
    for split_name in ("train", "validation", "test")
}

summary = pd.DataFrame(
    [
        {
            "split": split_name,
            "rows": len(frame),
            "start_month": frame[DATE_COLUMN].min(),
            "end_month": frame[DATE_COLUMN].max(),
            "companies": frame[ID_COLUMN].nunique(),
        }
        for split_name, frame in split_frames.items()
    ]
)

display(summary)
display(split_frames["train"].head())


Collapsing 548 duplicate rows across 548 (co_code, Month) keys by keeping the last occurrence per key.


,split,rows,start_month,end_month,companies
0,train,406415,2001-05-01,2019-12-01,3977
1,validation,51512,2020-01-01,2021-12-01,2631
2,test,114127,2022-01-01,2025-03-01,3686


,co_code,monthly_gross_return,Month,Momentum,log_ret,lag_ret,mktcap,price,equity_bv_on_stkdate,fy_book_value,...,is_missing_lag_ret_3,is_missing_lag_ret_6,is_missing_rolling_mean_3,is_missing_rolling_std_3,is_missing_rolling_mean_6,is_missing_rolling_std_6,is_missing_log_ret_lag_1,is_missing_log_ret_lag_3,is_missing_log_ret_lag_6,is_missing_month_index
0,1,0.994596,2016-09-01,-0.050237,0.086025,1.089834,110.8762,27.60,403.6,394.8,...,1,1,1,1,1,1,1,1,1,0
1,1,1.106926,2016-10-01,0.174515,-0.005419,0.994596,122.5262,30.50,403.6,394.8,...,1,1,0,1,0,1,0,1,1,0
2,1,0.801345,2016-11-01,0.175163,0.101586,1.106926,99.8288,24.85,403.6,394.8,...,1,1,0,0,0,0,0,1,1,0
3,1,1.145383,2016-12-01,-0.124685,-0.221463,0.801345,114.4917,28.50,403.6,394.8,...,0,1,0,0,0,0,0,0,1,0
4,1,1.003334,2017-01-01,0.273681,0.135739,1.145383,112.4831,28.00,406.2,394.8,...,0,1,0,0,0,0,0,0,1,0


## 3. Feature Construction + Split Verification

The assertions below make the split boundaries explicit and guard against time leakage.


In [3]:
duplicate_keys_remaining = int(panel.duplicated([ID_COLUMN, DATE_COLUMN]).sum())
if duplicate_keys_remaining > 0:
    print(
        f"Detected {duplicate_keys_remaining} duplicate (company, month) rows still present in `panel`. "
        "Re-collapsing duplicates and rebuilding split frames."
    )
    panel, duplicate_key_count, duplicate_rows_removed = collapse_duplicate_company_months(panel)
    split_frames = {
        split_name: panel.loc[panel[SOURCE_SPLIT_COLUMN] == split_name].copy().reset_index(drop=True)
        for split_name in ("train", "validation", "test")
    }
    duplicate_keys_remaining = int(panel.duplicated([ID_COLUMN, DATE_COLUMN]).sum())

train_end = split_frames["train"][DATE_COLUMN].max()
validation_start = split_frames["validation"][DATE_COLUMN].min()
validation_end = split_frames["validation"][DATE_COLUMN].max()
test_start = split_frames["test"][DATE_COLUMN].min()

assert train_end < validation_start, "Train and validation periods overlap."
assert validation_end < test_start, "Validation and test periods overlap."
assert duplicate_keys_remaining == 0, f"Found {duplicate_keys_remaining} duplicate (company, month) rows after collapse."

print(f"Train end month: {train_end:%Y-%m-%d}")
print(f"Validation window: {validation_start:%Y-%m-%d} -> {validation_end:%Y-%m-%d}")
print(f"Test start month: {test_start:%Y-%m-%d}")
print(f"Duplicate company-month keys collapsed: {duplicate_key_count}")
print(f"Duplicate rows removed before feature engineering: {duplicate_rows_removed}")
print(f"Engineered feature columns: {len(panel.columns)} total columns")


Train end month: 2019-12-01
Validation window: 2020-01-01 -> 2021-12-01
Test start month: 2022-01-01
Duplicate company-month keys collapsed: 548
Duplicate rows removed before feature engineering: 548
Engineered feature columns: 85 total columns


## 4. CatBoost Training

CatBoost is trained on the train split only. Validation is used for early stopping and model selection.


In [4]:
EXCLUDE_COLUMNS = {
    TARGET_COLUMN,
    ID_COLUMN,
    DATE_COLUMN,
    SOURCE_SPLIT_COLUMN,
    "cb_pred",
    "base_residual",
    "residual_target",
    "cnn_stage",
    "row_id",
    "cnn_resid_pred",
    "final_pred",
    "final_residual",
}

feature_columns = [column for column in panel.columns if column not in EXCLUDE_COLUMNS]
cat_features = [column for column in CAT_COLUMNS if column in feature_columns]


def make_feature_frame(frame: pd.DataFrame) -> pd.DataFrame:
    features = frame.reindex(columns=feature_columns).copy()
    for column in cat_features:
        features[column] = features[column].astype("string").fillna("__MISSING__")
    return features


def make_pool(frame: pd.DataFrame) -> Pool:
    return Pool(
        make_feature_frame(frame),
        label=frame[TARGET_COLUMN],
        cat_features=cat_features,
    )


train_pool = make_pool(split_frames["train"])
validation_pool = make_pool(split_frames["validation"])

cb_model = CatBoostRegressor(**CB_PARAMS)
try:
    cb_model.fit(train_pool, eval_set=validation_pool, use_best_model=True)
except CatBoostError as exc:
    print(f"GPU CatBoost failed ({exc}). Falling back to CPU.")
    cpu_params = dict(CB_PARAMS)
    cpu_params["task_type"] = "CPU"
    cb_model = CatBoostRegressor(**cpu_params)
    cb_model.fit(train_pool, eval_set=validation_pool, use_best_model=True)

for split_name, frame in split_frames.items():
    split_frames[split_name]["cb_pred"] = cb_model.predict(make_feature_frame(frame))
    split_frames[split_name]["base_residual"] = (
        split_frames[split_name][TARGET_COLUMN] - split_frames[split_name]["cb_pred"]
    )

split_frames["train"]["residual_target"] = np.nan
split_frames["validation"]["residual_target"] = split_frames["validation"]["base_residual"]
split_frames["test"]["residual_target"] = split_frames["test"]["base_residual"]

display(
    split_frames["validation"][
        [ID_COLUMN, DATE_COLUMN, TARGET_COLUMN, "cb_pred", "residual_target"]
    ].head()
)


0:	learn: 0.1922250	test: 0.2266816	best: 0.2266816 (0)	total: 40.1ms	remaining: 1m
100:	learn: 0.1189193	test: 0.1622942	best: 0.1622942 (100)	total: 1.88s	remaining: 26.1s
200:	learn: 0.0910018	test: 0.1232543	best: 0.1232543 (200)	total: 3.77s	remaining: 24.3s
300:	learn: 0.0781592	test: 0.1032763	best: 0.1032763 (300)	total: 5.62s	remaining: 22.4s
400:	learn: 0.0709888	test: 0.0916850	best: 0.0916850 (400)	total: 7.47s	remaining: 20.5s
500:	learn: 0.0667702	test: 0.0853776	best: 0.0853776 (500)	total: 9.35s	remaining: 18.6s
600:	learn: 0.0638147	test: 0.0810817	best: 0.0810817 (600)	total: 11.2s	remaining: 16.8s
700:	learn: 0.0615416	test: 0.0781211	best: 0.0781211 (700)	total: 13.1s	remaining: 14.9s
800:	learn: 0.0598713	test: 0.0762511	best: 0.0762511 (800)	total: 15s	remaining: 13.1s
900:	learn: 0.0584281	test: 0.0749566	best: 0.0749566 (900)	total: 16.9s	remaining: 11.2s
1000:	learn: 0.0571608	test: 0.0740098	best: 0.0740098 (1000)	total: 18.8s	remaining: 9.39s
1100:	learn: 0.0

,co_code,Month,monthly_gross_return,cb_pred,residual_target
0,5,2020-01-01,0.833573,1.014310,-0.180737
1,5,2020-02-01,0.877158,0.976850,-0.099692
2,5,2020-03-01,0.318207,0.873099,-0.554892
3,5,2020-04-01,1.332454,1.148772,0.183681
4,5,2020-05-01,0.840833,0.946155,-0.105322


## 5. Residual Computation + CNN Dataset Construction

The CNN is trained only on validation residuals. To keep that stage causal and at least partially out-of-sample, the validation period is split chronologically into a CNN-training portion and a CNN-holdout portion.


In [5]:
validation_months = np.sort(split_frames["validation"][DATE_COLUMN].dropna().unique())
if len(validation_months) < 2:
    raise ValueError("Validation split needs at least two months for CNN train/holdout separation.")

cut_idx = int(len(validation_months) * CNN_TRAIN_FRACTION)
cut_idx = min(max(cut_idx, 1), len(validation_months) - 1)
cnn_train_months = set(validation_months[:cut_idx])
cnn_holdout_months = set(validation_months[cut_idx:])

split_frames["train"]["cnn_stage"] = "history_only"
split_frames["validation"]["cnn_stage"] = np.where(
    split_frames["validation"][DATE_COLUMN].isin(cnn_train_months),
    "cnn_train",
    "cnn_holdout",
)
split_frames["test"]["cnn_stage"] = "test"

panel = pd.concat(
    [split_frames["train"], split_frames["validation"], split_frames["test"]],
    ignore_index=True,
    sort=False,
)
panel = panel.sort_values([ID_COLUMN, DATE_COLUMN, SOURCE_SPLIT_COLUMN], kind="mergesort").reset_index(drop=True)
panel["row_id"] = panel.index


def encode_for_cnn(df: pd.DataFrame, columns: list[str], categorical_columns: list[str]):
    encoded = pd.DataFrame(index=df.index)
    category_maps = {}
    for column in columns:
        series = df[column] if column in df.columns else pd.Series(np.nan, index=df.index)
        if column in categorical_columns:
            categories = pd.Index(series.astype("string").fillna("__MISSING__").unique())
            category_maps[column] = categories.tolist()
            encoded[column] = pd.Categorical(
                series.astype("string").fillna("__MISSING__"),
                categories=categories,
            ).codes.astype("float32")
        else:
            encoded[column] = pd.to_numeric(series, errors="coerce").astype("float32")
    return encoded, category_maps


cnn_base_features, cnn_category_maps = encode_for_cnn(panel, feature_columns, cat_features)
cnn_feature_frame = cnn_base_features.copy()
cnn_feature_frame["cb_pred"] = panel["cb_pred"].astype("float32")
cnn_feature_columns = list(cnn_base_features.columns) + ["cb_pred"]

scaler_mask = (
    (panel[SOURCE_SPLIT_COLUMN] == "train")
    | ((panel[SOURCE_SPLIT_COLUMN] == "validation") & (panel["cnn_stage"] == "cnn_train"))
)
scaler_values = cnn_feature_frame.loc[scaler_mask, cnn_feature_columns].to_numpy(dtype=np.float32)
scaler_mean = np.nanmean(scaler_values, axis=0)
scaler_std = np.nanstd(scaler_values, axis=0)
scaler_std[~np.isfinite(scaler_std) | (scaler_std < 1e-6)] = 1.0


def normalize_window(window: np.ndarray) -> np.ndarray:
    window = (window - scaler_mean) / scaler_std
    return np.nan_to_num(window, nan=0.0, posinf=0.0, neginf=0.0)


def build_sequence_dataset(
    panel_df: pd.DataFrame,
    feature_df: pd.DataFrame,
    eligible_mask: pd.Series,
    seq_len: int,
    target_col: str,
):
    sequences = []
    targets = []
    row_ids = []
    ordered = panel_df.sort_values([ID_COLUMN, DATE_COLUMN, "row_id"], kind="mergesort")

    for _, group in ordered.groupby(ID_COLUMN, sort=False):
        group_row_ids = group["row_id"].to_numpy()
        group_features = feature_df.loc[group_row_ids, cnn_feature_columns].to_numpy(dtype=np.float32)
        group_targets = group[target_col].to_numpy(dtype=np.float32)

        for end in range(seq_len - 1, len(group)):
            row_id = int(group_row_ids[end])
            if not bool(eligible_mask.loc[row_id]):
                continue
            window = group_features[end - seq_len + 1 : end + 1]
            sequences.append(normalize_window(window).T)
            targets.append(group_targets[end])
            row_ids.append(row_id)

    if not sequences:
        raise ValueError("No sequences were created. Reduce SEQ_LEN or check split coverage.")

    return (
        np.stack(sequences).astype(np.float32),
        np.asarray(targets, dtype=np.float32),
        np.asarray(row_ids, dtype=np.int64),
    )


val_train_mask = (
    (panel[SOURCE_SPLIT_COLUMN] == "validation")
    & (panel["cnn_stage"] == "cnn_train")
    & panel["residual_target"].notna()
)
val_holdout_mask = (
    (panel[SOURCE_SPLIT_COLUMN] == "validation")
    & (panel["cnn_stage"] == "cnn_holdout")
    & panel["residual_target"].notna()
)
test_mask = (
    (panel[SOURCE_SPLIT_COLUMN] == "test")
    & panel["residual_target"].notna()
)

X_cnn_train, y_cnn_train, row_id_train = build_sequence_dataset(
    panel,
    cnn_feature_frame,
    val_train_mask,
    SEQ_LEN,
    "residual_target",
)
X_cnn_val, y_cnn_val, row_id_val = build_sequence_dataset(
    panel,
    cnn_feature_frame,
    val_holdout_mask,
    SEQ_LEN,
    "residual_target",
)
X_cnn_test, y_cnn_test, row_id_test = build_sequence_dataset(
    panel,
    cnn_feature_frame,
    test_mask,
    SEQ_LEN,
    "residual_target",
)

print(f"CNN train sequences: {X_cnn_train.shape}")
print(f"CNN validation holdout sequences: {X_cnn_val.shape}")
print(f"CNN test sequences: {X_cnn_test.shape}")


CNN train sequences: (39128, 82, 6)
CNN validation holdout sequences: (11470, 82, 6)
CNN test sequences: (109305, 82, 6)


## 6. CNN Training

The CNN learns residual corrections using only causal feature windows.


In [6]:
class ResidualSequenceDataset(Dataset):
    def __init__(self, features: np.ndarray, targets: np.ndarray):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.targets = torch.tensor(targets, dtype=torch.float32)

    def __len__(self) -> int:
        return len(self.targets)

    def __getitem__(self, index: int):
        return self.features[index], self.targets[index]


class ResidualCNN(nn.Module):
    def __init__(self, input_channels: int):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv1d(input_channels, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(64, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.head = nn.Linear(32, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.backbone(x).squeeze(-1)
        return self.head(x).squeeze(-1)


def make_loader(features: np.ndarray, targets: np.ndarray, shuffle: bool) -> DataLoader:
    dataset = ResidualSequenceDataset(features, targets)
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=PIN_MEMORY,
    )


def mse_on_loader(model: nn.Module, loader: DataLoader) -> float:
    model.eval()
    total_loss = 0.0
    total_items = 0
    criterion = nn.MSELoss()
    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(DEVICE, non_blocking=True)
            batch_y = batch_y.to(DEVICE, non_blocking=True)
            preds = model(batch_x)
            loss = criterion(preds, batch_y)
            total_loss += float(loss.item()) * len(batch_y)
            total_items += len(batch_y)
    return total_loss / max(total_items, 1)


train_loader = make_loader(X_cnn_train, y_cnn_train, shuffle=True)
val_loader = make_loader(X_cnn_val, y_cnn_val, shuffle=False)

cnn_model = ResidualCNN(input_channels=X_cnn_train.shape[1]).to(DEVICE)
optimizer = torch.optim.Adam(cnn_model.parameters(), lr=CNN_LR, weight_decay=WEIGHT_DECAY)
criterion = nn.MSELoss()

best_state = None
best_val_loss = float("inf")
history = []

for epoch in range(1, CNN_EPOCHS + 1):
    cnn_model.train()
    train_loss_total = 0.0
    train_items = 0

    for batch_x, batch_y in train_loader:
        batch_x = batch_x.to(DEVICE, non_blocking=True)
        batch_y = batch_y.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        preds = cnn_model(batch_x)
        loss = criterion(preds, batch_y)
        loss.backward()
        optimizer.step()

        train_loss_total += float(loss.item()) * len(batch_y)
        train_items += len(batch_y)

    train_loss = train_loss_total / max(train_items, 1)
    val_loss = mse_on_loader(cnn_model, val_loader)
    history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})
    print(f"Epoch {epoch:02d} | train_loss={train_loss:.6f} | val_loss={val_loss:.6f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(cnn_model.state_dict())

if best_state is not None:
    cnn_model.load_state_dict(best_state)

history_df = pd.DataFrame(history)
display(history_df.tail())


Epoch 01 | train_loss=0.006193 | val_loss=0.005898
Epoch 02 | train_loss=0.004375 | val_loss=0.005663
Epoch 03 | train_loss=0.004128 | val_loss=0.005426
Epoch 04 | train_loss=0.003990 | val_loss=0.005069
Epoch 05 | train_loss=0.003912 | val_loss=0.005203
Epoch 06 | train_loss=0.003882 | val_loss=0.005241
Epoch 07 | train_loss=0.003846 | val_loss=0.004958
Epoch 08 | train_loss=0.003808 | val_loss=0.004848
Epoch 09 | train_loss=0.003807 | val_loss=0.004741
Epoch 10 | train_loss=0.003737 | val_loss=0.004821
Epoch 11 | train_loss=0.003747 | val_loss=0.004781
Epoch 12 | train_loss=0.003643 | val_loss=0.004762
Epoch 13 | train_loss=0.003641 | val_loss=0.004721
Epoch 14 | train_loss=0.003597 | val_loss=0.004662
Epoch 15 | train_loss=0.003560 | val_loss=0.004637
Epoch 16 | train_loss=0.003527 | val_loss=0.004633
Epoch 17 | train_loss=0.003520 | val_loss=0.004493
Epoch 18 | train_loss=0.003525 | val_loss=0.004562
Epoch 19 | train_loss=0.003429 | val_loss=0.004474
Epoch 20 | train_loss=0.003418 

,epoch,train_loss,val_loss
25,26,0.003210,0.004375
26,27,0.003190,0.004306
27,28,0.003195,0.004234
28,29,0.003159,0.004342
29,30,0.003125,0.004485


## 7. Evaluation + Metrics

CatBoost metrics are reported on train, validation, and test. The combined model is reported on the validation holdout and on test rows that have enough history for the sequence model.


In [7]:
def batched_predict(model: nn.Module, features: np.ndarray) -> np.ndarray:
    model.eval()
    outputs = []
    loader = make_loader(features, np.zeros(len(features), dtype=np.float32), shuffle=False)
    with torch.no_grad():
        for batch_x, _ in loader:
            batch_x = batch_x.to(DEVICE, non_blocking=True)
            preds = model(batch_x).detach().cpu().numpy()
            outputs.append(preds)
    return np.concatenate(outputs, axis=0)


panel["cnn_resid_pred"] = np.nan
panel.loc[row_id_train, "cnn_resid_pred"] = batched_predict(cnn_model, X_cnn_train)
panel.loc[row_id_val, "cnn_resid_pred"] = batched_predict(cnn_model, X_cnn_val)
panel.loc[row_id_test, "cnn_resid_pred"] = batched_predict(cnn_model, X_cnn_test)
panel["final_pred"] = panel["cb_pred"] + panel["cnn_resid_pred"].fillna(0.0)
panel["final_residual"] = panel[TARGET_COLUMN] - panel["final_pred"]


def regression_metrics(frame: pd.DataFrame, pred_col: str) -> dict:
    valid = frame[[TARGET_COLUMN, pred_col]].dropna()
    if valid.empty:
        return {"rows": 0, "RMSE": np.nan, "MAE": np.nan, "R2": np.nan, "Directional_Accuracy": np.nan}
    y_true = valid[TARGET_COLUMN].to_numpy(dtype=float)
    y_pred = valid[pred_col].to_numpy(dtype=float)
    residual = y_true - y_pred
    rmse = float(np.sqrt(np.mean(residual ** 2)))
    mae = float(np.mean(np.abs(residual)))
    sst = float(np.sum((y_true - y_true.mean()) ** 2))
    r2 = float(1.0 - np.sum(residual ** 2) / sst) if sst > 0 else np.nan
    directional_accuracy = float(np.mean(np.sign(y_true) == np.sign(y_pred)))
    return {
        "rows": int(len(valid)),
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "Directional_Accuracy": directional_accuracy,
    }


def monthly_ic(frame: pd.DataFrame, pred_col: str) -> dict:
    ic_values = []
    for _, group in frame.groupby(DATE_COLUMN):
        valid = group[[TARGET_COLUMN, pred_col]].dropna()
        if len(valid) < 2:
            continue
        corr = valid[TARGET_COLUMN].rank().corr(valid[pred_col].rank(), method="pearson")
        if pd.notna(corr):
            ic_values.append(float(corr))
    if not ic_values:
        return {"IC_mean": np.nan, "IC_std": np.nan, "IC_months": 0}
    return {
        "IC_mean": float(np.mean(ic_values)),
        "IC_std": float(np.std(ic_values)),
        "IC_months": int(len(ic_values)),
    }


def top_k_monthly_return(frame: pd.DataFrame, pred_col: str, k: int = TOP_K) -> dict:
    monthly_returns = []
    for _, group in frame.groupby(DATE_COLUMN):
        valid = group[[TARGET_COLUMN, pred_col]].dropna().sort_values(pred_col, ascending=False).head(k)
        if valid.empty:
            continue
        monthly_returns.append(float(valid[TARGET_COLUMN].mean()))
    if not monthly_returns:
        return {"TopK_Return": np.nan, "Sharpe_Proxy": np.nan}
    topk_mean = float(np.mean(monthly_returns))
    topk_std = float(np.std(monthly_returns))
    sharpe_proxy = topk_mean / topk_std if topk_std > 0 else np.nan
    return {"TopK_Return": topk_mean, "Sharpe_Proxy": sharpe_proxy}


def mean_company_autocorr(frame: pd.DataFrame, residual_col: str) -> float:
    autocorr_values = []
    ordered = frame.sort_values([ID_COLUMN, DATE_COLUMN], kind="mergesort")
    for _, group in ordered.groupby(ID_COLUMN, sort=False):
        series = group[residual_col].dropna()
        if len(series) < 2 or series.std() == 0:
            continue
        autocorr_values.append(float(series.autocorr(lag=1)))
    return float(np.nanmean(autocorr_values)) if autocorr_values else np.nan


def metric_row(name: str, frame: pd.DataFrame, pred_col: str) -> dict:
    row = {"slice": name}
    row.update(regression_metrics(frame, pred_col))
    row.update(monthly_ic(frame, pred_col))
    row.update(top_k_monthly_return(frame, pred_col, k=TOP_K))
    return row


train_eval = panel.loc[panel[SOURCE_SPLIT_COLUMN] == "train"].copy()
validation_eval = panel.loc[panel[SOURCE_SPLIT_COLUMN] == "validation"].copy()
validation_holdout_eval = panel.loc[
    (panel[SOURCE_SPLIT_COLUMN] == "validation") & (panel["cnn_stage"] == "cnn_holdout")
].copy()
test_eval = panel.loc[panel[SOURCE_SPLIT_COLUMN] == "test"].copy()
test_sequence_eval = test_eval.loc[test_eval["cnn_resid_pred"].notna()].copy()

metrics_df = pd.DataFrame(
    [
        metric_row("train_catboost", train_eval, "cb_pred"),
        metric_row("validation_catboost", validation_eval, "cb_pred"),
        metric_row("test_catboost", test_eval, "cb_pred"),
        metric_row("validation_holdout_combined", validation_holdout_eval, "final_pred"),
        metric_row("test_combined", test_sequence_eval, "final_pred"),
    ]
)

base_residual_variance = float(test_sequence_eval["base_residual"].var())
final_residual_variance = float(test_sequence_eval["final_residual"].var())
if np.isfinite(base_residual_variance) and base_residual_variance > 0:
    variance_reduction = float(1.0 - final_residual_variance / base_residual_variance)
else:
    variance_reduction = np.nan

residual_diagnostics = pd.DataFrame(
    [
        {
            "slice": "test_sequence_rows",
            "base_residual_variance": base_residual_variance,
            "final_residual_variance": final_residual_variance,
            "variance_reduction": variance_reduction,
            "base_residual_autocorr_lag1": mean_company_autocorr(test_sequence_eval, "base_residual"),
            "final_residual_autocorr_lag1": mean_company_autocorr(test_sequence_eval, "final_residual"),
        }
    ]
)

display(metrics_df)
display(residual_diagnostics)


,slice,rows,RMSE,MAE,R2,Directional_Accuracy,IC_mean,IC_std,IC_months,TopK_Return,Sharpe_Proxy
0,train_catboost,406415,0.052736,0.032853,0.925939,0.999995,0.921990,0.031837,224,1.732772,5.182054
1,validation_catboost,51512,0.071095,0.038551,0.899788,0.999961,0.922776,0.020132,24,1.958229,6.745558
2,test_catboost,114127,0.058798,0.034820,0.871908,0.999982,0.896930,0.020887,39,1.797674,9.744849
3,validation_holdout_combined,11798,0.064783,0.034062,0.919312,1.000000,0.918832,0.017365,5,2.096324,7.125249
4,test_combined,109305,0.059820,0.036520,0.865745,0.999982,0.903219,0.022333,39,1.785248,9.754673


,slice,base_residual_variance,final_residual_variance,variance_reduction,base_residual_autocorr_lag1,final_residual_autocorr_lag1
0,test_sequence_rows,0.003426,0.003285,0.041011,-0.08351,-0.049801


## 8. Inference Pipeline

For new rows at time `t`, append them after the historical panel, rebuild leak-safe features, predict with CatBoost, then pass sequence windows through the CNN to get the residual correction.


In [8]:
def encode_for_cnn_with_maps(
    df: pd.DataFrame,
    columns: list[str],
    categorical_columns: list[str],
    category_maps: dict,
) -> pd.DataFrame:
    encoded = pd.DataFrame(index=df.index)
    for column in columns:
        series = df[column] if column in df.columns else pd.Series(np.nan, index=df.index)
        if column in categorical_columns:
            categories = category_maps[column]
            codes = pd.Categorical(
                series.astype("string").fillna("__MISSING__"),
                categories=categories,
            ).codes.astype("float32")
            codes = np.where(codes < 0, np.nan, codes)
            encoded[column] = codes
        else:
            encoded[column] = pd.to_numeric(series, errors="coerce").astype("float32")
    return encoded


def build_inference_sequences(
    panel_df: pd.DataFrame,
    feature_df: pd.DataFrame,
    eligible_mask: pd.Series,
    seq_len: int,
):
    sequences = []
    row_ids = []
    ordered = panel_df.sort_values([ID_COLUMN, DATE_COLUMN, "row_id"], kind="mergesort")

    for _, group in ordered.groupby(ID_COLUMN, sort=False):
        group_row_ids = group["row_id"].to_numpy()
        group_features = feature_df.loc[group_row_ids, cnn_feature_columns].to_numpy(dtype=np.float32)
        for end in range(seq_len - 1, len(group)):
            row_id = int(group_row_ids[end])
            if not bool(eligible_mask.loc[row_id]):
                continue
            window = group_features[end - seq_len + 1 : end + 1]
            sequences.append(normalize_window(window).T)
            row_ids.append(row_id)

    if not sequences:
        return np.empty((0, len(cnn_feature_columns), seq_len), dtype=np.float32), np.array([], dtype=np.int64)

    return np.stack(sequences).astype(np.float32), np.asarray(row_ids, dtype=np.int64)


def predict_pipeline(history_df: pd.DataFrame, new_df: pd.DataFrame) -> pd.DataFrame:
    history = history_df.copy()
    history[SOURCE_SPLIT_COLUMN] = "history"
    incoming = new_df.copy()
    incoming[SOURCE_SPLIT_COLUMN] = "inference"

    combined = pd.concat([history, incoming], ignore_index=True, sort=False)
    combined = drop_previous_engineering(combined)
    combined = coerce_panel_types(combined)
    combined, _, _ = collapse_duplicate_company_months(combined)
    combined = engineer_features(combined)
    combined = add_missing_indicators(combined)
    combined = combined.sort_values([ID_COLUMN, DATE_COLUMN], kind="mergesort").reset_index(drop=True)
    combined["row_id"] = combined.index

    catboost_features = combined.reindex(columns=feature_columns).copy()
    for column in cat_features:
        catboost_features[column] = catboost_features[column].astype("string").fillna("__MISSING__")
    combined["cb_pred"] = cb_model.predict(catboost_features)

    cnn_encoded = encode_for_cnn_with_maps(combined, feature_columns, cat_features, cnn_category_maps)
    cnn_features = cnn_encoded.copy()
    cnn_features["cb_pred"] = combined["cb_pred"].astype("float32")
    cnn_features = cnn_features.reindex(combined.index)

    eligible_mask = combined[SOURCE_SPLIT_COLUMN].eq("inference")
    X_new, row_ids_new = build_inference_sequences(combined, cnn_features, eligible_mask, SEQ_LEN)

    combined["cnn_resid_pred"] = 0.0
    if len(row_ids_new):
        combined.loc[row_ids_new, "cnn_resid_pred"] = batched_predict(cnn_model, X_new)
    combined["final_pred"] = combined["cb_pred"] + combined["cnn_resid_pred"]

    return combined.loc[
        combined[SOURCE_SPLIT_COLUMN] == "inference",
        [ID_COLUMN, DATE_COLUMN, "cb_pred", "cnn_resid_pred", "final_pred"],
    ].copy()


artifacts = {
    "catboost_model": cb_model,
    "cnn_model": cnn_model,
    "feature_columns": feature_columns,
    "cnn_feature_columns": cnn_feature_columns,
    "sequence_length": SEQ_LEN,
}

artifacts


{'catboost_model': CatBoostRegressor(depth=10, eval_metric='RMSE', iterations=1500, l2_leaf_reg=3.0, learning_rate=0.05, loss_function='RMSE', random_seed=42, task_type='GPU', verbose=100),
 'cnn_model': ResidualCNN(
   (backbone): Sequential(
     (0): Conv1d(82, 64, kernel_size=(3,), stride=(1,), padding=(1,))
     (1): ReLU()
     (2): Conv1d(64, 32, kernel_size=(3,), stride=(1,), padding=(1,))
     (3): ReLU()
     (4): AdaptiveAvgPool1d(output_size=1)
   )
   (head): Linear(in_features=32, out_features=1, bias=True)
 ),
 'feature_columns': ['Momentum',
  'log_ret',
  'lag_ret',
  'mktcap',
  'price',
  'equity_bv_on_stkdate',
  'fy_book_value',
  'lag_mv',
  'BM_sep',
  'Size_Label',
  'BM_Label',
  'Year',
  'sa_net_worth',
  'sa_total_assets',
  'sa_sales',
  'sa_cost_of_goods_sold',
  'sa_total_interest_exp',
  'sa_selling_distribution_exp',
  'Month_annual',
  'Corrected_Year',
  'Corrected_Month',
  'lagged_book_equity',
  'lagged_assets',
  'lagged2_assets',
  'OpProf',
  'I